In [ ]:
import pandas as pd

matches = pd.read_csv(r"/content/matches (1).csv")
deliveries = pd.read_csv(r"/content/deliveries (2).csv")

matches.head()
deliveries.head()

In [ ]:
deliveries.columns

In [ ]:
pp_data = deliveries[deliveries['over'] <= 6]

In [ ]:
deliveries['is_wicket'] = deliveries['player_dismissed'].notnull().astype(int)

pp_data = deliveries[deliveries['over'] <= 6]

pp_summary = pp_data.groupby(['match_id', 'batting_team']).agg({
    'total_runs': 'sum',
    'is_wicket': 'sum'
}).reset_index()

In [ ]:
pp_summary.rename(columns={
    'total_runs': 'current_score',
    'is_wicket': 'wickets_lost'
}, inplace=True)

In [ ]:
df = pp_summary.merge(
    matches[['id', 'winner']],
    left_on='match_id',
    right_on='id'
)

In [ ]:
df['win'] = (df['batting_team'] == df['winner']).astype(int)

In [ ]:
df = df[['batting_team', 'current_score', 'wickets_lost', 'win']]

df.head()

In [ ]:
df['run_rate'] = df['current_score'] / 6

In [ ]:
df['pressure'] = df['current_score'] / (df['wickets_lost'] + 1)

In [ ]:
df.columns

In [ ]:
X = df.drop('win', axis=1)
y = df['win']

In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
model.predict_proba([[60, 1, 10, 30] + [0]*len(X.columns[4:])])

In [ ]:
import pandas as pd

sample = pd.DataFrame([X.iloc[0].values], columns=X.columns)
model.predict_proba(sample)

In [ ]:
import pandas as pd

importance = pd.Series(model.coef_[0], index=X.columns)
importance.sort_values(ascending=False).head(10)

In [ ]:
import plotly.express as px

fig = px.histogram(df, x='current_score', color='win',
                   title="Score vs Win Distribution")

fig.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

In [ ]:
import pickle

pickle.dump(model, open("ipl_model.pkl", "wb"))